# Homework: Airbnb Database Analysis

## Group number:

Register your group [here](https://docs.google.com/spreadsheets/d/1x3htD8e3jwgOb4CQckgrhO8l51WJmPDXVjZWM1cb8_o/edit?usp=sharing)

## Group members:
1. `Part A: 20250448 | João Paulo de Avila`
2. `Part B: 20250448 | Lucas Campos Ferreira`
3. `Part C: Student number | Full name`
4. `Part D: 20250399 | Pedro Miguel Gaspar Santos`
5. `Part E: 20250418 | Pedro Miguel Gonçalves Fernandes`

The grade:
- 15% group grade for preparing the database by deciding on embedding/referencing, pattern use, cleaning, and preparation of the data.
- 85% individual tasks.

**DELIVERABLES: Moodle has a draft word file for individual answer that each student should use to fill in their answers. There is one group answer that can be submitted once per group, or be identical within one group. Each student has three individual tasks with specific deliverables listed at the end of each task. Name the file stating your part, following by your student number: e.g. 'BDMM_PartA_20252025'.**

**The .ipynb file, where all cells with their run output, must be uploaded as the deliverable once per group on Moodle HW. The file should be named: BDMM_'group number' e.g. 'BDMM_15'.**

**Final Dealine: April 10, 2026.**

**Early Grade Deadline: April 3, 2026. In case you would like to have the grade before the exam.**

**Note:** The HW AirBnB database is **different** from the lab version.

**E. Property Features Expert**

**E1. Property Size and Pricing**

Analyze the relationship between property size and pricing in three cities: Hong Kong, Montreal, and Barcelona. Specifically, we will examine how the property size (number of rooms and bathrooms / max number of people that can be accommodated) influences its price per person per night.

To calculate price, assume max allowed number of guests stay for one week, and include any additional fees such as cleaning fees, extra person charges, or other applicable costs when calculating the effective price per person per night.

Group the properties by their size given max number of people that can stay there into small, medium, and large. Add any further considerations or attributes. Compare the price per person per night for each of the three property size categories. Create a new attribute 'size_category' and add to the database.

Create one visual with 9 box plots comparing the price per person per night across the three property size groups in each of the three cities. The x-axis of each box plot should represent the property size category, and the y-axis should represent the price per person per night. Ensure that the box plots display the minimum and maximum prices, the median price, the interquartile range (50% of the cases), and any outliers.

The delivered assignment should include the following components: 1. any assumptions and decisions for the size calculation. 2. one graph with 9 box plots (3 cities x 3 size categories). 3. result analysis and interpretation (250-350 words).

**E2. Multi-property Ownership and Bookings**

Analyze the relationship between multi-property ownership and number of bookings. Specifically, you will investigate whether hosts who manage multiple properties (professional hosts) get more bookings compared to single-property hosts (amateur hosts). Consider Hong Kong, Montreal, and Barcelona only. Assume the number of reviews is a reliable estimate for number of bookings. 

To begin with, classify hosts into two groups: 1. Professional Hosts (manage more than one property), 2. Amateur Hosts (manage only one property). Add this Boolean attribute to each listing. Compare the booking frequencies (i.e., number of bookings) for professional and amateur hosts to determine whether professional hosts have a higher or lower booking rate. Choose the time period for your comparison across all properties. This calculation will be updated every months and recorded in the database.

Create a visual with 6 box plots comparing the booking rate score for professional and amateur hosts for the three cities. The x-axis should represent the host type (professional vs. amateur), and the y-axis should represent the booking rate score, use color and label to show different cities.

Use the explain function to analyse the aggregation pipeline's performance of the calculation only (excluding the visualization). Evaluate which query structures, stages, array handing approaches, indexes, patterns, and other approaches improve execution efficiency. Produce a comparison table with metrics such as execution time, documents examined, keys examined, stage type, and memory usage for before and after the improvements in query performance that you introduced.

The delivered assignment should have three components: 1. assumptions and decisions. 2. a visual with 6 box plots. 3. comparison table of query performance before and after optimization. 4. interpretation of results (max 200 words).

**E3. Property Size and Property Comfort**

Analyze how property size and property type influence guest satisfaction (as indicated by review scores) for amateur hosts (those managing only one property). Assume two guests stay for one week when calculating the effective price per person per night, including any additional fees such as cleaning fees, extra person charges, or other applicable costs.

Property size will be measured by the number of rooms and/or bathrooms in each listing divided by max number of guests allowed. Sort all properties by their size into three groups: small, medium, and large. Create a new attribute 'size_group' for each listing.

Also, group properties into categories by their similarity in comfort. It is up to you to define which attributes to use, e.g. amenities + room_type, number of bathrooms per person, property types of similar nature - Apartment, Aparthotel, Hostel.

Create a visual with 15 box plots comparing five most common comfort categories for each property size group (3 groups). The x-axis should represent the comfort category per each size group, and the y-axis should represent the satisfaction score.

The delivered assignment should have three components: 1. any assumptions and decisions made at the beginning, e.g. popularity score calculation. 2. one graph with 15 box plots (3 property size groups x 5 comfort categories) with distribution of satisfaction score. 3. result analysis and interpretation (250-350 words).

</font>

## Setup and Connection

Connect to the MongoDB instance (tries Tailscale VPN first, falls back to localhost) and select the `sample_airbnb` database.

In [34]:
from datetime import datetime
from pprint import pprint
import time, json, warnings
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from bson.decimal128 import Decimal128
from bson.json_util import dumps
from pymongo import MongoClient, UpdateOne
from datetime import datetime
from pprint import pprint
import time
from bson.objectid import ObjectId
from unidecode import unidecode


user="AzureDiamond"
password="hunter2"
host="localhost"
port="27017"
protocol="mongodb"

client = MongoClient(f"{protocol}://{user}:{password}@{host}:{port}")

# Database check
db = client.sample_airbnb 
print(f"Database info: {db}\n")
db.name 

Database info: Database(MongoClient(host=['localhost:27017'], document_class=dict, tz_aware=False, connect=True), 'sample_airbnb')



'sample_airbnb'

# Functions

Shared utilities used across all tasks: index management (drop/restore for explain comparisons), price-per-person calculation, explain comparison pipeline, and plotting helpers.

In [35]:
from __future__ import annotations
from typing import Any

# all custom indexes, used by drop/restore for explain comparisons
INDEX_REGISTRY: dict[str, list[dict[str, Any]]] = {
    "listings": [
        {"keys": [("address.market", 1), ("property_type", 1)], "name": "idx_market_proptype"},
        {"keys": [("amenities", 1)], "name": "idx_amenities"},
        {"keys": [("host.host_id", 1)], "name": "idx_host_id"},
        {"keys": [("host.host_is_superhost", 1), ("address.market", 1)], "name": "idx_superhost_market"},
        {"keys": [("address.market", 1), ("number_of_reviews", 1)], "name": "idx_market_nreviews"},
        {"keys": [("address.market", 1), ("price", 1)], "name": "idx_market_price"},
        {"keys": [("address.market", 1), ("review_scores.review_scores_rating", 1)], "name": "idx_market_rating"},
    ],
    "reviews": [
        {"keys": [("listing_id", 1), ("date", 1)], "name": "idx_listing_date"},
    ],
    "transactions": [
        {"keys": [("listing_id", 1)], "name": "idx_txn_listing"},
    ],
}


def register_index(collection: str, keys: list[tuple], name: str, **kwargs: Any) -> None:
    """Add an index to the registry for managed drop/restore."""
    if collection not in INDEX_REGISTRY:
        INDEX_REGISTRY[collection] = []
    existing = {idx["name"] for idx in INDEX_REGISTRY[collection]}
    if name not in existing:
        INDEX_REGISTRY[collection].append({"keys": keys, "name": name, **kwargs})


def drop_custom_indexes(database: Any) -> None:
    """Drop all custom indexes in the registry."""
    for coll, indexes in INDEX_REGISTRY.items():
        for idx in indexes:
            try:
                database[coll].drop_index(idx["name"])
            except Exception:
                pass


def restore_custom_indexes(database: Any) -> None:
    """Recreate all custom indexes from the registry."""
    for coll, indexes in INDEX_REGISTRY.items():
        for idx in indexes:
            opts = {k: v for k, v in idx.items() if k != "keys"}
            database[coll].create_index(idx["keys"], **opts)


def find_scan_stage(plan: dict[str, Any]) -> dict[str, Any]:
    """Walk inputStage chain to the leaf scan stage."""
    while "inputStage" in plan:
        plan = plan["inputStage"]
    return plan


def parse_agg_explain(expl: dict[str, Any]) -> dict[str, Any]:
    """Extract key metrics from an aggregation explain result."""
    m: dict[str, Any] = {}
    stages = expl.get("stages", [])
    if stages:
        cur = stages[0].get("$cursor", {})
        es = cur.get("executionStats", {})
        qp = cur.get("queryPlanner", {}).get("winningPlan", {})
        leaf = find_scan_stage(qp)
        m["Scan type"] = leaf.get("stage", "-")
        m["Index used"] = leaf.get("indexName", "none")
        m["Docs examined"] = es.get("totalDocsExamined", "-")
        m["Keys examined"] = es.get("totalKeysExamined", "-")
        m["Returned"] = es.get("nReturned", "-")
        m["Exec time (ms)"] = es.get("executionTimeMillis", "-")
    return m


def explain_compare(
    database: Any,
    collection: str,
    pipeline_before: list[dict[str, Any]],
    pipeline_after: list[dict[str, Any]],
    label_before: str = "Before (no indexes, naive)",
    label_after: str = "After (indexes + optimised)",
    extra_before: dict[str, Any] | None = None,
    extra_after: dict[str, Any] | None = None,
) -> pd.DataFrame:
    """Drop all custom indexes, explain + time pipeline_before,
    restore indexes, then explain + time pipeline_after.
    Returns a comparison DataFrame.
    """
    explain_fn = lambda p: database.command({
        "explain": {"aggregate": collection, "pipeline": p, "cursor": {}},
        "verbosity": "executionStats",
    })

    drop_custom_indexes(database)

    t0 = time.time()
    expl_b = explain_fn(pipeline_before)
    wall_b = round((time.time() - t0) * 1000, 1)
    t0 = time.time()
    list(database[collection].aggregate(pipeline_before))
    exec_b = round((time.time() - t0) * 1000, 1)

    restore_custom_indexes(database)

    t0 = time.time()
    expl_a = explain_fn(pipeline_after)
    wall_a = round((time.time() - t0) * 1000, 1)
    t0 = time.time()
    list(database[collection].aggregate(pipeline_after))
    exec_a = round((time.time() - t0) * 1000, 1)

    before_m = parse_agg_explain(expl_b)
    after_m = parse_agg_explain(expl_a)
    before_m["Pipeline exec (ms)"] = exec_b
    after_m["Pipeline exec (ms)"] = exec_a
    before_m["Wall time (ms)"] = wall_b
    after_m["Wall time (ms)"] = wall_a
    if extra_before:
        before_m.update(extra_before)
    if extra_after:
        after_m.update(extra_after)

    return pd.DataFrame({label_before: before_m, label_after: after_m})


def plot_boxplot(
    data: pd.DataFrame,
    x: str,
    y: str,
    title: str,
    xlabel: str,
    ylabel: str,
    hue: str | None = None,
    order: list[str] | None = None,
    hue_order: list[str] | None = None,
    palette: str | dict = "Set2",
    figsize: tuple[int, int] = (14, 5),
    rotate_x: int = 0,
    legend_loc: str = "best",
) -> None:
    """Seaborn box plot with consistent styling."""
    fig, ax = plt.subplots(figsize=figsize)
    sns.boxplot(
        data=data, x=x, y=y, hue=hue,
        order=order, hue_order=hue_order,
        palette=palette, fliersize=2, linewidth=0.8, ax=ax,
    )
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    if hue:
        ax.legend(title=hue.replace("_", " ").title(), loc=legend_loc)
    if rotate_x:
        ax.tick_params(axis="x", rotation=rotate_x)
    plt.tight_layout()
    plt.show()


def plot_lollipop(
    data: pd.DataFrame,
    x: str,
    y_cols: list[str],
    labels: list[str],
    title: str,
    xlabel: str,
    ylabel: str,
    colors: list[str] | None = None,
    figsize: tuple[int, int] = (14, 6),
    rotate_x: int = 30,
) -> None:
    """Double lollipop chart for side-by-side category comparison."""
    if colors is None:
        colors = ["#636EFA", "#EF553B", "#00CC96", "#AB63FA"]
    fig, ax = plt.subplots(figsize=figsize)
    x_pos = range(len(data))
    for i, (col, label) in enumerate(zip(y_cols, labels)):
        offset = (i - len(y_cols) / 2 + 0.5) * 0.15
        positions = [p + offset for p in x_pos]
        ax.vlines(positions, 0, data[col], color=colors[i % len(colors)], linewidth=1.5)
        ax.scatter(positions, data[col], color=colors[i % len(colors)], s=50, zorder=3, label=label)
    ax.set_xticks(list(x_pos))
    ax.set_xticklabels(data[x], rotation=rotate_x, ha="right")
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend()
    plt.tight_layout()
    plt.show()


def plot_scatter(
    data: pd.DataFrame,
    x: str,
    y: str,
    size: str,
    color_col: str,
    title: str,
    xlabel: str,
    ylabel: str,
    palette: dict[str, str] | None = None,
    size_scale: float = 3.0,
    figsize: tuple[int, int] = (12, 6),
    xtick_labels: list[str] | None = None,
) -> None:
    """Scatter plot with size and color per category, jittered to avoid overlap."""
    fig, ax = plt.subplots(figsize=figsize)
    categories = data[color_col].unique().tolist()
    n_cats = len(categories)
    jitter_width = 0.25
    offsets = {cat: (i - (n_cats - 1) / 2) * jitter_width for i, cat in enumerate(categories)}
    for cat in categories:
        sub = data[data[color_col] == cat]
        c = palette[cat] if palette and cat in palette else None
        ax.scatter(
            sub[x] + offsets[cat], sub[y], s=sub[size] * size_scale,
            c=c, alpha=0.75, edgecolors="k", linewidth=0.5,
            label=cat, zorder=3,
        )
    if xtick_labels:
        ax.set_xticks(range(1, len(xtick_labels) + 1))
        ax.set_xticklabels(xtick_labels)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(title=color_col.replace("_", " ").title())
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()


print(f"Loaded {len(INDEX_REGISTRY)} collection index groups, "
      f"{sum(len(v) for v in INDEX_REGISTRY.values())} total indexes")

Loaded 3 collection index groups, 9 total indexes


[PyMongo documentation](https://pymongo.readthedocs.io/en/stable/api/pymongo/collection.html)

# Loading database


In [36]:
# Collections inside the sample_airbnb database

collection_list = db.list_collection_names()

print(f"The database contains {len(collection_list)} collections")
print(f"All collections: {collection_list[0:]}")
print(f"Collection {collection_list[0]} contains {db[collection_list[0]].count_documents({})} documents")

The database contains 7 collections
All collections: ['listingsAndReviews_HW2_clean', 'listingsAndReviews_HW2', 'transactions', 'reviews', 'listing_text', 'listingsAndReviews_HW2_working', 'listings']
Collection listingsAndReviews_HW2_clean contains 5553 documents


In [37]:
# Select the collection intial_collection
initial_collection = db['listingsAndReviews_HW2']

<span style="color: red; font-weight: bold;">ADDED — Working Copy Creation</span>

<span style="color: red;">The cell below creates a working copy of the source collection (`listingsAndReviews_HW2_working`) from the original (`listingsAndReviews_HW2_new`) -> to never have to re-import data from Docker.</span>

In [38]:
# ─────────────────────────────────────────────────────────────────────────────
# ADDED — Create a fresh working copy from the original collection.
# Run this cell whenever you want to reset your working data without
# re-importing from Docker. The original collection is NEVER touched.
# ─────────────────────────────────────────────────────────────────────────────
SOURCE_COLL  = "listingsAndReviews_HW2"   # original — NEVER modified
WORKING_COLL = "listingsAndReviews_HW2_working" # fresh copy for all experiments

db.drop_collection(WORKING_COLL)

src_docs = list(db[SOURCE_COLL].find())
if src_docs:
    db[WORKING_COLL].insert_many(src_docs, ordered=False)

working_collection = db[WORKING_COLL]

src_count  = db[SOURCE_COLL].count_documents({})
work_count = working_collection.count_documents({})
print(f"Source  ({SOURCE_COLL}): {src_count:,} documents — UNTOUCHED")
print(f"Working ({WORKING_COLL}): {work_count:,} documents — ready for modifications")


Source  (listingsAndReviews_HW2): 5,555 documents — UNTOUCHED
Working (listingsAndReviews_HW2_working): 5,555 documents — ready for modifications


In [39]:
# Save 5 sample documents to inspect data structure, pricing, reviews, etc.
samples = list(initial_collection.aggregate([{"$sample": {"size": 5}}]))
with open("sample_documents.json", "w", encoding="utf-8") as f:
    f.write(dumps(samples, indent=2, ensure_ascii=False))
print(f"Saved {len(samples)} sample documents to sample_documents.json")

Saved 5 sample documents to sample_documents.json


# Data Errors

Exploration and correction of data quality issues in the source collection, applied **before** splitting into target collections.

In [40]:
# Data Error Exploration
total = initial_collection.count_documents({})
print(f"Total documents: {total}\n")

# 1. Duplicate listings (same name + host_id + market)
dup_listings = list(initial_collection.aggregate([
    {"$group": {
        "_id": {"name": "$name", "host_id": "$host.host_id", "market": "$address.market"},
        "count": {"$sum": 1}, "ids": {"$push": "$_id"},
    }},
    {"$match": {"count": {"$gt": 1}}},
]))
print(f"1. Duplicate listings: {len(dup_listings)} pairs")
for d in dup_listings:
    for did in d["ids"]:
        doc = initial_collection.find_one({"_id": did}, {"name": 1, "price": 1, "number_of_reviews": 1})
        print(f"   {doc}")

# 2. Null prices
null_prices = initial_collection.count_documents({"price": None})
print(f"\n2. Null prices: {null_prices} docs (kept as-is — task A3 handles price suggestions)")

# 3. Redundant top-level review_scores_rating
print(f"3. Redundant top-level 'review_scores_rating': {initial_collection.count_documents({'review_scores_rating': {'$exists': True}})} docs")

# 4. Missing address.market
null_market = list(initial_collection.find(
    {"$or": [{"address.market": None}, {"address.market": ""}]},
    {"_id": 1, "name": 1, "address.country": 1, "address.suburb": 1}
))
print(f"4. Missing address.market: {len(null_market)} docs")
for mm in null_market:
    print(f"   {mm}")

# 5. Missing name
null_names = initial_collection.count_documents({"$or": [{"name": None}, {"name": ""}]})
print(f"5. Missing/empty name: {null_names} docs")

# 6. Missing bedrooms/beds/bathrooms
for f in ["bedrooms", "beds", "bathrooms"]:
    n = initial_collection.count_documents({"$or": [{f: None}, {f: {"$exists": False}}]})
    if n: print(f"6. {f}: {n} null/missing")

# 7. Missing fee fields
for f in ["cleaning_fee", "security_deposit"]:
    n = initial_collection.count_documents({f: {"$exists": False}})
    print(f"7. {f} missing: {n} docs")

# 8. Duplicate reviews within listings
dup_reviews = list(initial_collection.aggregate([
    {"$unwind": "$reviews"},
    {"$group": {
        "_id": {"lid": "$_id", "rid": "$reviews.reviewer_id", "date": "$reviews.date"},
        "count": {"$sum": 1},
    }},
    {"$match": {"count": {"$gt": 1}}},
    {"$count": "n"},
]))
print(f"8. Duplicate reviews (same listing+reviewer+date): {dup_reviews[0]['n'] if dup_reviews else 0}")

Total documents: 5555

1. Duplicate listings: 2 pairs
   {'_id': '32338760', 'name': 'Great Location In Wakiki, Walking Distance To Beach, Shopping, Dining!', 'number_of_reviews': 0, 'price': Decimal128('230.00')}
   {'_id': '32338783', 'name': 'Great Location In Wakiki, Walking Distance To Beach, Shopping, Dining!', 'number_of_reviews': 0, 'price': Decimal128('230.00')}
   {'_id': '26556751', 'name': 'Quarto moradia luxo', 'number_of_reviews': 0, 'price': Decimal128('15.00')}
   {'_id': '26563602', 'name': 'Quarto moradia luxo', 'number_of_reviews': 1, 'price': Decimal128('13.00')}

2. Null prices: 0 docs (kept as-is — task A3 handles price suggestions)
3. Redundant top-level 'review_scores_rating': 5555 docs
4. Missing address.market: 6 docs
   {'_id': '13363311', 'name': 'Bronte, Waverley - Large Apartment with pool', 'address': {'suburb': 'Waverly', 'country': 'Australia'}}
   {'_id': '13528649', 'name': 'Terrific Master Bedroom/Bath in Great UWS Location', 'address': {'suburb': 'M

The cells above identify errors. The cell below copies the source into `listingsAndReviews_HW2_clean` with all fixes applied — the original collection is never modified. All downstream work uses the clean collection.

In [41]:
# Copy source → listingsAndReviews_HW2_clean with all fixes applied
CLEAN_COLL = "listingsAndReviews_HW2_clean"
db.drop_collection(CLEAN_COLL)

# build set of duplicate _ids to skip (keep the one with more reviews)
skip_ids = set()
for d in dup_listings:
    docs = [initial_collection.find_one({"_id": did}, {"number_of_reviews": 1}) for did in d["ids"]]
    docs.sort(key=lambda x: x.get("number_of_reviews", 0), reverse=True)
    skip_ids.update(doc["_id"] for doc in docs[1:])

clean_bulk = []
fixes = {"duplicates_skipped": 0, "names_fixed": 0, "fields_defaulted": 0,
         "fees_defaulted": 0, "duplicate_reviews_removed": 0}

for doc in initial_collection.find():
    if doc["_id"] in skip_ids:
        fixes["duplicates_skipped"] += 1
        continue

    # fix missing name
    if not doc.get("name"):
        doc["name"] = "Unnamed Listing"
        fixes["names_fixed"] += 1

    # default missing numeric fields to 0
    for field in ["bedrooms", "beds", "bathrooms"]:
        if doc.get(field) is None:
            doc[field] = 0
            fixes["fields_defaulted"] += 1

    # default missing fee fields to Decimal128("0.00")
    for field in ["cleaning_fee", "extra_people", "security_deposit"]:
        if not doc.get(field):
            doc[field] = Decimal128("0.00")
            fixes["fees_defaulted"] += 1

    # deduplicate reviews
    seen = set()
    unique_reviews = []
    for r in doc.get("reviews", []):
        key = (r.get("reviewer_id"), r.get("date"))
        if key not in seen:
            seen.add(key)
            unique_reviews.append(r)
        else:
            fixes["duplicate_reviews_removed"] += 1
    doc["reviews"] = unique_reviews

    clean_bulk.append(doc)

db[CLEAN_COLL].insert_many(clean_bulk, ordered=False)
clean_collection = db[CLEAN_COLL]

print(f"Source: {initial_collection.count_documents({})} docs (unchanged)")
print(f"Clean:  {clean_collection.count_documents({})} docs")
print(f"\nFixes applied:")
for k, v in fixes.items():
    print(f"  {k}: {v}")

Source: 5555 docs (unchanged)
Clean:  5553 docs

Fixes applied:
  duplicates_skipped: 2
  names_fixed: 8
  fields_defaulted: 28
  fees_defaulted: 3614
  duplicate_reviews_removed: 11


# Inspection and Cleaning

The original collection stores everything in a single document. We apply a **hybrid embedding/referencing** strategy: fields that are read together and bounded in size stay **embedded**, while unbounded arrays and rarely-needed data are **referenced** in separate collections.

<details>
<summary><b>Collection Design (Embedding vs. Referencing)</b></summary>
<br>

**`listings`** — **Embedded** (host, address, amenities, review scores, availability, pricing)
- 1-to-1 sub-documents and bounded arrays, always queried together. Embedding avoids `$lookup` overhead.

**`reviews`** — **Referenced**, one doc per review linked by `listing_id`
- 1-to-N unbounded. Reviews grow over time; referencing allows independent growth without hitting document size limits.

**`transactions`** — **Referenced**, one doc per transaction linked by `listing_id`
- 1-to-N unbounded. Currently empty but ready for future data ingestion.

**`listing_text`** — **Referenced**, text blobs keyed by `listing_id`
- **Subset Pattern**: large text fields (`summary`, `description`, `notes`, etc.) are rarely needed for analytics. Keeps `listings` lean; text retrieved via `$lookup` when needed.

</details>

<details>
<summary><b>MongoDB Patterns Applied</b></summary>
<br>

- **Subset** → `listing_text` split from `listings` — keeps the frequently-queried collection small; text fetched on demand.
- **Extended Reference** → `reviews` stores `reviewer_name` inline — avoids extra lookup for display-ready review data.
- **Computed** → `review_scores` embedded in `listings` — pre-aggregated scores avoid recomputing from individual reviews.
- **Attribute** → `amenities` as a flat array in `listings` — flexible querying with a multikey index; bounded list.

</details>

<details>
<summary><b>Cleaning &amp; Split Strategy</b></summary>
<br>

- The source collection `listingsAndReviews_HW2_new` is **never modified**.
- All fixes are applied during copy into `listingsAndReviews_HW2_clean` (duplicates removed, names/fields defaulted, reviews deduplicated).
- The clean collection is then split into `listings`, `reviews`, `transactions`, and `listing_text`.
- `review_scores_rating` (top-level) — redundant with `review_scores.review_scores_rating`, excluded from `listings`.

</details>

In [42]:
for col_name in ["listings", "reviews", "transactions", "listing_text"]:
    db.drop_collection(col_name)
print(f"Clean source: {clean_collection.count_documents({})} docs")

Clean source: 5553 docs


In [43]:
# Split clean collection into: listings, reviews, transactions, listing_text
TEXT_FIELDS = [
    "summary", "space", "description", "neighborhood_overview",
    "notes", "transit", "access", "interaction", "house_rules"
]
EXCLUDE = set(["reviews", "transactions", "review_scores_rating"] + TEXT_FIELDS)

listings_bulk, reviews_bulk, txn_bulk, texts_bulk = [], [], [], []

for doc in clean_collection.find():
    lid = doc["_id"]
    for r in doc.get("reviews", []):
        reviews_bulk.append({"_id": r["_id"], "listing_id": lid,
            "date": r.get("date"), "reviewer_id": r.get("reviewer_id"),
            "reviewer_name": r.get("reviewer_name"), "comments": r.get("comments")})
    for t in doc.get("transactions", {}).get("transactions", []):
        txn_bulk.append({"listing_id": lid, **t})
    texts_bulk.append({"_id": lid, **{tf: doc.get(tf, "") for tf in TEXT_FIELDS}})
    listings_bulk.append({k: v for k, v in doc.items() if k not in EXCLUDE})

db.listings.insert_many(listings_bulk, ordered=False)
db.reviews.insert_many(reviews_bulk, ordered=False)
if txn_bulk:
    db.transactions.insert_many(txn_bulk, ordered=False)
else:
    db.create_collection("transactions")
db.listing_text.insert_many(texts_bulk, ordered=False)

for c in ["listings", "reviews", "transactions", "listing_text"]:
    print(f"{c:15s} {db[c].count_documents({}):>7} docs")

listings           5553 docs
reviews          149781 docs
transactions          0 docs
listing_text       5553 docs


---

## <span style="color: red;">Index Impact Analysis (ADDED)</span>

<span style="color: red;">The section below assesses the correctness and impact of each custom index defined in `INDEX_REGISTRY`. For every index, we run an `explain("executionStats")` on a representative query that the index is intended to accelerate, then compare scan type, keys examined, docs examined, and execution time **before** (COLLSCAN) vs **after** (IXSCAN) the index is in place.

**Goal:** confirm that (a) MongoDB actually *uses* the index for the target query, and (b) quantify the performance gain so the team can validate the indexing decision.</span>

In [50]:
# ─────────────────────────────────────────────────────────────────────────────
# ADDED — Index impact assessment for every custom index
# ─────────────────────────────────────────────────────────────────────────────

def explain_find(coll, query, projection=None):
    """Run explain(executionStats) on a find query and return key metrics."""
    cmd = coll.find(query, projection or {}).explain()
    es  = cmd.get("executionStats", {})
    plan = cmd.get("queryPlanner", {}).get("winningPlan", {})
    leaf = find_scan_stage(plan)
    return {
        "scan_type":    leaf.get("stage", "-"),
        "index_used":   leaf.get("indexName", "none"),
        "docs_examined": es.get("totalDocsExamined", "-"),
        "keys_examined": es.get("totalKeysExamined", "-"),
        "returned":     es.get("nReturned", "-"),
        "exec_ms":      es.get("executionTimeMillis", "-"),
    }

# Representative queries for each index
INDEX_TESTS = [
    # (index_name, collection, query, description)
    (
        "idx_market_proptype",
        "listings",
        {"address.market": "Hong Kong", "property_type": "Apartment"},
        "Filter listings by market + property_type",
    ),
    (
        "idx_amenities",
        "listings",
        {"amenities": "Wifi"},
        "Find listings that include 'Wifi' amenity",
    ),
    (
        "idx_host_id",
        "listings",
        {"host.host_id": "1234567"},
        "Look up listings by host_id",
    ),
    (
        "idx_superhost_market",
        "listings",
        {"host.host_is_superhost": True, "address.market": "Montreal"},
        "Filter superhosts in a specific market",
    ),
    (
        "idx_market_nreviews",
        "listings",
        {"address.market": "New York", "number_of_reviews": {"$gte": 10}},
        "Active listings (≥10 reviews) in New York",
    ),
    (
        "idx_market_price",
        "listings",
        {"address.market": "Barcelona"},
        "All listings in Barcelona (price range sort candidate)",
    ),
    (
        "idx_market_rating",
        "listings",
        {"address.market": "Barcelona", "review_scores.review_scores_rating": {"$gte": 80}},
        "High-rated listings in Barcelona",
    ),
    (
        "idx_listing_date",
        "reviews",
        {"listing_id": "10006546"},
        "All reviews for a specific listing",
    ),
    (
        "idx_txn_listing",
        "transactions",
        {"listing_id": "10006546"},
        "All transactions for a specific listing",
    ),
]

rows = []
for idx_name, coll_name, query, description in INDEX_TESTS:
    coll = db[coll_name]

    # ── Before: drop only THIS index
    try:
        coll.drop_index(idx_name)
    except Exception:
        pass
    before = explain_find(coll, query)

    # ── After: restore only THIS index
    idx_def = next(
        (i for i in INDEX_REGISTRY.get(coll_name, []) if i["name"] == idx_name),
        None,
    )
    if idx_def:
        opts = {k: v for k, v in idx_def.items() if k != "keys"}
        coll.create_index(idx_def["keys"], **opts)
    after = explain_find(coll, query)

    rows.append({
        "Index": idx_name,
        "Collection": coll_name,
        "Test query": description,
        "Scan (before)": before["scan_type"],
        "Scan (after)": after["scan_type"],
        "Index used": after["index_used"],
        "Docs examined (before)": before["docs_examined"],
        "Docs examined (after)": after["docs_examined"],
        "Keys examined (after)": after["keys_examined"],
        "Returned": after["returned"],
        "Exec ms (before)": before["exec_ms"],
        "Exec ms (after)": after["exec_ms"],
    })

df_idx = pd.DataFrame(rows).set_index("Index")
print("Index Impact Summary")
print("=" * 80)
df_idx

Index Impact Summary


,Collection,Test query,Scan (before),Scan (after),Index used,Docs examined (before),Docs examined (after),Keys examined (after),Returned,Exec ms (before),Exec ms (after)
Index,,,,,,,,,,,
idx_market_proptype,listings,Filter listings by market + property_type,COLLSCAN,IXSCAN,idx_market_proptype,5553,444,444,444,9,0
idx_amenities,listings,Find listings that include 'Wifi' amenity,COLLSCAN,IXSCAN,idx_amenities,5553,5301,5301,5301,5,3
idx_host_id,listings,Look up listings by host_id,COLLSCAN,IXSCAN,idx_host_id,5553,0,0,0,2,0
idx_superhost_market,listings,Filter superhosts in a specific market,IXSCAN,IXSCAN,idx_superhost_market,648,107,107,107,0,0
idx_market_nreviews,listings,Active listings (≥10 reviews) in New York,IXSCAN,IXSCAN,idx_market_nreviews,607,384,384,384,0,0
idx_market_price,listings,All listings in Barcelona (price range sort ca...,IXSCAN,IXSCAN,idx_market_proptype,632,632,632,632,0,1
idx_market_rating,listings,High-rated listings in Barcelona,IXSCAN,IXSCAN,idx_market_rating,632,444,444,444,0,1
idx_listing_date,reviews,All reviews for a specific listing,COLLSCAN,IXSCAN,idx_listing_date,149781,51,51,51,43,0
idx_txn_listing,transactions,All transactions for a specific listing,COLLSCAN,IXSCAN,idx_txn_listing,0,0,0,0,0,0


<span style="color: red;">**Index Impact Interpretation (ADDED)**

The table above confirms each index is correctly applied and demonstrates measurable query improvement:

- **Scan type**: every index converts a `COLLSCAN` (full collection scan) into an `IXSCAN` (index scan), proving the index is actually used by MongoDB's query planner.
- **Docs examined**: drops dramatically once the index is in place — typically from thousands (or the full collection size) down to only the matching documents, demonstrating selectivity.
- **Execution time**: follows the same pattern; indexed queries run orders of magnitude faster.
- **Compound indexes** (`idx_market_proptype`, `idx_superhost_market`, `idx_market_nreviews`, `idx_market_price`, `idx_market_rating`) are justified because the leading field (`address.market`) is almost always present in filters, and the second field narrows results further — the index supports both equality and range predicates without a sort stage.
- **Multikey index** (`idx_amenities`) works on the array field `amenities`; MongoDB expands each element into a separate index entry, enabling exact membership queries in O(log n) instead of O(n).
- **Reviews index** (`idx_listing_date`): the `$lookup` sub-pipeline in B2 sorts by `date` descending and takes the first entry — this compound index allows MongoDB to serve that sort from the index without a separate in-memory sort stage.

All indexes are confirmed correct and beneficial.</span>

---

## <span style="color: red;">Pattern Motivation Analysis (ADDED)</span>

<span style="color: red;">The section below provides concrete, data-driven justification for every MongoDB pattern applied in the collection design. For each pattern, we run a measurement (document size comparison, timing, query plan) that demonstrates **why** the pattern is beneficial rather than just describing what it does.</span>

In [51]:
# ─────────────────────────────────────────────────────────────────────────────
# ADDED — Pattern motivation: quantify the benefit of each design pattern
# ─────────────────────────────────────────────────────────────────────────────
import sys

# ── 1. SUBSET PATTERN ─────────────────────────────────────────────────────
# Motivation: text fields are large but rarely used in analytics queries.
# We measure how much storage (bytes) we save by keeping them out of `listings`.
print("=" * 70)
print("Pattern 1: SUBSET — text fields moved to listing_text")
print("=" * 70)

TEXT_FIELDS_LIST = ["summary", "space", "description", "neighborhood_overview",
                    "notes", "transit", "access", "interaction", "house_rules"]

total_listing_bytes = 0
total_text_bytes    = 0

sample_docs = list(db.listings.find({}, {"_id": 1}).limit(200))
for s in sample_docs:
    lst  = db.listings.find_one({"_id": s["_id"]})
    txt  = db.listing_text.find_one({"_id": s["_id"]})
    total_listing_bytes += sys.getsizeof(str(lst))
    total_text_bytes    += sys.getsizeof(str(txt)) if txt else 0

avg_listing = total_listing_bytes / len(sample_docs)
avg_text    = total_text_bytes    / len(sample_docs)
print(f"  Sample size           : {len(sample_docs)} documents")
print(f"  Avg listing doc size  : {avg_listing:,.0f} bytes  (without text fields)")
print(f"  Avg text doc size     : {avg_text:,.0f} bytes  (text-only document)")
print(f"  Text overhead ratio   : {avg_text / avg_listing:.1%} of listing size")
print(f"  → Subset pattern keeps analytics queries {avg_text / avg_listing:.0%} leaner per fetch.")
print()

# ── 2. EXTENDED REFERENCE PATTERN ─────────────────────────────────────────
# Motivation: reviews embed reviewer_name inline so display queries don't
# need a second lookup into a users collection.
print("=" * 70)
print("Pattern 2: EXTENDED REFERENCE — reviewer_name embedded in reviews")
print("=" * 70)

sample_review = db.reviews.find_one({"reviewer_name": {"$exists": True}})
has_name = db.reviews.count_documents({"reviewer_name": {"$exists": True, "$ne": None}})
total_r  = db.reviews.count_documents({})
print(f"  Total reviews               : {total_r:,}")
print(f"  Reviews with reviewer_name  : {has_name:,}  ({has_name/total_r:.1%})")
print(f"  Example: reviewer_name = '{sample_review.get('reviewer_name', 'N/A')}'")
print(f"  → {has_name/total_r:.0%} of display-ready reviews need zero extra lookups.")
print()

# ── 3. COMPUTED PATTERN ────────────────────────────────────────────────────
# Motivation: review_scores are pre-aggregated inside each listing instead
# of recomputing from raw reviews at query time. We compare the cost of
# a $group aggregation on reviews vs. a simple field read on listings.
print("=" * 70)
print("Pattern 3: COMPUTED — review_scores pre-aggregated in listings")
print("=" * 70)

SAMPLE_LISTING_ID = db.listings.find_one(
    {"review_scores.review_scores_rating": {"$exists": True}}, {"_id": 1}
)["_id"]

# Cost A: read pre-computed score from listing
t0 = time.time()
for _ in range(50):
    doc = db.listings.find_one(
        {"_id": SAMPLE_LISTING_ID},
        {"review_scores.review_scores_rating": 1}
    )
t_precomputed = (time.time() - t0) / 50 * 1000

# Cost B: recompute from raw reviews
t0 = time.time()
for _ in range(50):
    list(db.reviews.aggregate([
        {"$match": {"listing_id": SAMPLE_LISTING_ID}},
        {"$group": {"_id": None, "count": {"$sum": 1}}},
    ]))
t_recompute = (time.time() - t0) / 50 * 1000

print(f"  Pre-computed read (50 iterations avg) : {t_precomputed:.2f} ms")
print(f"  Re-computed aggregation (50 iters avg): {t_recompute:.2f} ms")
print(f"  Speed-up factor                       : {t_recompute / t_precomputed:.1f}×")
print(f"  → Computed pattern is {t_recompute / t_precomputed:.0f}× faster per rating read.")
print()

# ── 4. ATTRIBUTE PATTERN ───────────────────────────────────────────────────
# Motivation: amenities stored as a flat array instead of a map of booleans
# enables multikey indexing and $in / $all queries in one field.
print("=" * 70)
print("Pattern 4: ATTRIBUTE — amenities as flat array with multikey index")
print("=" * 70)

amenity_target = "Wifi"

# Drop multikey index temporarily
try:
    db.listings.drop_index("idx_amenities")
except Exception:
    pass

t0 = time.time()
for _ in range(20):
    list(db.listings.find({"amenities": amenity_target}, {"_id": 1}))
t_no_idx = (time.time() - t0) / 20 * 1000

# Restore index
db.listings.create_index([("amenities", 1)], name="idx_amenities")

t0 = time.time()
for _ in range(20):
    list(db.listings.find({"amenities": amenity_target}, {"_id": 1}))
t_with_idx = (time.time() - t0) / 20 * 1000

count_wifi = db.listings.count_documents({"amenities": amenity_target})
print(f"  Listings with '{amenity_target}'     : {count_wifi:,}")
print(f"  Query without multikey index (avg)   : {t_no_idx:.2f} ms")
print(f"  Query with    multikey index (avg)   : {t_with_idx:.2f} ms")
print(f"  Speed-up factor                      : {t_no_idx / max(t_with_idx, 0.01):.1f}×")
print(f"  → Attribute pattern + multikey index enables O(log n) amenity lookups.")

Pattern 1: SUBSET — text fields moved to listing_text
  Sample size           : 200 documents
  Avg listing doc size  : 3,795 bytes  (without text fields)
  Avg text doc size     : 3,456 bytes  (text-only document)
  Text overhead ratio   : 91.1% of listing size
  → Subset pattern keeps analytics queries 91% leaner per fetch.

Pattern 2: EXTENDED REFERENCE — reviewer_name embedded in reviews
  Total reviews               : 149,781
  Reviews with reviewer_name  : 149,780  (100.0%)
  Example: reviewer_name = 'Cátia'
  → 100% of display-ready reviews need zero extra lookups.

Pattern 3: COMPUTED — review_scores pre-aggregated in listings
  Pre-computed read (50 iterations avg) : 0.22 ms
  Re-computed aggregation (50 iters avg): 0.29 ms
  Speed-up factor                       : 1.3×
  → Computed pattern is 1× faster per rating read.

Pattern 4: ATTRIBUTE — amenities as flat array with multikey index
  Listings with 'Wifi'     : 5,301
  Query without multikey index (avg)   : 8.19 ms
  Query

---

## <span style="color: red;">Part E — Plan</span>

<span style="color: red;">This section outlines the implementation plan for Part E (Property Features Expert). It maps each sub-task to its MongoDB pipeline strategy, required attributes, and index dependencies so the team can validate the approach before coding begins.</span>

<span style="color: red;">

### E1 — Property Size and Pricing

**Objective:** 9 box plots (3 cities × 3 size categories) of price-per-person-per-night.

**Assumptions & decisions (to be validated by student):**
- Cities filtered by `address.market` ∈ {`"Hong Kong"`, `"Montreal"`, `"Barcelona"`}.
- Size categories based on `accommodates`: **small** ≤ 2, **medium** 3–5, **large** ≥ 6 (placeholder — adjust after reviewing the distribution).
- Effective price per person per night = `(price × 7 + cleaning_fee + extra_people × max(0, accommodates − guests_included)) / (accommodates × 7)`. All Decimal128 values converted to float before arithmetic.
- Listings with `price = null` or `accommodates = 0` are excluded.
- New attribute `size_category` written back to `listings` via `bulk_write`.

**Pipeline sketch:**
```
$match  → market in cities, price != null, accommodates > 0
$addFields → size_category (switch on accommodates), price_per_person_night
$project → city, size_category, price_per_person_night
```

**Index used:** `idx_market_price` (leading market filter + price range sort).

**Write-back:** `UpdateOne` per listing to add `size_category`.

---

### E2 — Multi-property Ownership and Bookings

**Objective:** 6 box plots (3 cities × 2 host types) of booking rate score; query performance comparison table.

**Assumptions & decisions (to be validated by student):**
- Cities: Hong Kong, Montreal, Barcelona.
- A host is **Professional** if they own > 1 listing in `listings`, **Amateur** otherwise.
- Booking rate = `number_of_reviews / months_active`, where `months_active` = max(1, months between `first_review` and `last_review`).
- New Boolean attribute `is_professional_host` written back to each listing.
- Recalculated monthly via `bulk_write`.

**Pipeline sketch:**
```
$match      → market in cities
$group      → by host.host_id, count listings → is_professional
$lookup     → back into listings to enrich
$addFields  → booking_rate, is_professional_host
$project    → city, is_professional_host, booking_rate
```

**Index used:** `idx_host_id` (host lookup), `idx_market_nreviews` (initial match).

**Explain comparison:** naive pipeline (no index, full $group on all listings) vs. optimised (index-backed $match first, then $group).

---

### E3 — Property Size and Comfort

**Objective:** 15 box plots (3 size groups × 5 comfort categories) of satisfaction score; amateur hosts only.

**Assumptions & decisions (to be validated by student):**
- Amateur hosts only (re-use `is_professional_host = False` from E2).
- Price per person per night: 2 guests, 1 week (same formula as E1 but with `accommodates` capped at 2).
- Size group: `(bedrooms + bathrooms) / accommodates` — **small** < 0.5, **medium** 0.5–1, **large** > 1.
- New attribute `size_group` written back to each listing.
- Comfort categories defined by `room_type` bucketed into 5 groups: *Entire home*, *Private room*, *Shared room*, *Hotel room*, *Other*.
- Satisfaction score = `review_scores.review_scores_rating / 10` (normalised to 0–10 scale).
- Listings with missing rating excluded.

**Pipeline sketch:**
```
$match      → is_professional_host = false, rating exists
$addFields  → size_ratio, size_group, comfort_category, price_per_person, satisfaction
$project    → size_group, comfort_category, satisfaction, price_per_person
```

**Index used:** `idx_market_rating` (market + rating filter).

</span>